# Query Expansion using LLMs with CoT Prompt

## Install Dependencies

In [1]:
!pip install bitsandbytes==0.43.1
!pip install transformers==4.40.2
!pip install peft==0.11.1
!pip install accelerate==0.28.0
!pip install sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

## Imports

In [2]:
import os
import json
import gzip
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


## Load Zephyr or Similar LLM

In [3]:
model_name = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
)
llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

model-00008-of-00008.safetensors:   0%|          | 0.00/816M [00:00<?, ?B/s]

model-00007-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0


## Load MSMARCO Queries and Candidate Docs

In [9]:
def load_tsv_gz(url, out_path):
    import requests
    response = requests.get(url)
    if response.status_code == 200:
        with open(out_path, "wb") as f:
            f.write(response.content)
    else:
        raise Exception(f"Failed to download from {url} — status code {response.status_code}")

import os, gzip

# Ensure directory
os.makedirs("data", exist_ok=True)

# Download files
load_tsv_gz(
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz",
    "data/queries.tsv.gz"
)
load_tsv_gz(
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz",
    "data/top1000.tsv.gz"
)

# Parse queries
queries = {}
with gzip.open("data/queries.tsv.gz", "rt") as f:
    for line in f:
        qid, text = line.strip().split("\t")
        queries[qid] = text


## Define CoT Prompt Template

In [10]:
def cot_prompt(query):
    return f"User Query: {query} Break down the query into its subtopics and reformulate it to be more informative for document retrieval. Do not repeat the query.\nExpanded Query:"


## Generate Expanded Queries using LLM

In [11]:
expanded_queries = {}
for qid, query in tqdm(list(queries.items())[:43]):
    prompt = cot_prompt(query)
    output = llm_pipeline(prompt, max_new_tokens=64, do_sample=False)[0]["generated_text"]
    # Extract only the expanded query after the last "Expanded Query:"
    expanded = output.split("Expanded Query:")[-1].strip()
    expanded_queries[qid] = expanded


100%|██████████| 43/43 [14:42<00:00, 20.53s/it]


## Save Expanded Queries to JSON

In [12]:
with open("expanded_queries.json", "w") as f:
    json.dump(expanded_queries, f, indent=2)


# Save expanded queries to Google Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import json
# Save the expanded queries to your Google Drive
with open('/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/expanded_queries.json', 'w') as f:
    json.dump(expanded_queries, f, indent=2)

print("Expanded queries saved to Google Drive in 'MyDrive/expanded_queries.json'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Expanded queries saved to Google Drive in 'MyDrive/expanded_queries.json'
